In [26]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [27]:
load_dotenv(override=True)
openrouter_api = os.getenv("OPENROUTER_API_KEY")
groq_api = os.getenv("GROQ_API_KEY")

if openrouter_api:
    display("Openrouter api key exists")
else:
    display("Opnerouter api key doesnt exists")
if groq_api:
    display("groq api key exists")
else:
    display("groq api key doesnt exists")
    
    

'Openrouter api key exists'

'groq api key exists'

In [28]:


groq = OpenAI(base_url="https://api.groq.com/openai/v1", api_key= groq_api)

openrouter = OpenAI(base_url="https://openrouter.ai/api/v1" , api_key= openrouter_api)

ollama = OpenAI(base_url="http://localhost:11434/v1", api_key= "ollama")

In [7]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a stock trader"},
]

In [9]:
response = openrouter.chat.completions.create(
    model="google/gemma-4-31b-it:free",
    messages= tell_a_joke
)
display(Markdown(response.choices[0].message.content))

A stock trader is walking down the street when he sees a friend who looks absolutely miserable.

"What's wrong?" the trader asks.

The friend sighs and says, "My wife told me that if I didn't stop obsessing over my portfolio, she was going to leave me."

The trader looks concerned and asks, "That's terrible! What are you going to do?"

The friend replies, "I don't know. I'm just waiting for the market to bounce back so I can afford the divorce lawyer."

### SOLVING PUZZLE USING DIFFERENT MODELS , OLLAMA, GROQ, OPENROUTER

In [11]:
easy_puzzle = [
    {"role": "user", "content": 
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

In [15]:
response = ollama.chat.completions.create(model="llama3.1:8b", messages=easy_puzzle)
display(Markdown(response.choices[0].message.content))


0.5

In [20]:
response = groq.chat.completions.create(model="llama-3.3-70b-versatile", messages=easy_puzzle )
display(Markdown(response.choices[0].message.content))


1/2

In [ ]:
response = openrouter.chat.completions.create(model="openai/gpt-oss-120b:free", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))


2/3

## LANGCHAIN

In [30]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=groq_api)
response = llm.invoke(easy_puzzle)

display(Markdown(response.content))



1/2

## Dynamic Switching between MODELS

In [36]:
from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq

USE_LOCAL_MODEL = False

if USE_LOCAL_MODEL:
    llm = ChatOllama(model="gemma4:e4b")
else:
    llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=groq_api)

response = llm.invoke(easy_puzzle)

display(Markdown(response.content))

1/2

## Conversation between two models

In [37]:
groq_model="llama-3.3-70b-versatile"
openrouter_model = "openai/gpt-oss-120b:free"

groq_system = "You are a chatbot who is very seductive in a snarky way. If other person is not seductive, you try to seduce them and keep chatting."

openrouter_system = "You are a very polite, courteous chatbot. If the other person is seductive, \
you try to calm them down and keep chatting."

groq_messages = ["Hi there"]
openrouter_messages = ["Hi"]

In [40]:
def call_groq():
    messages = [{"role": "system", "content": groq_system}]

    for g_msg, o_msg in zip(groq_messages, openrouter_messages):
        messages.append({"role": "assistant", "content": g_msg})
        messages.append({"role": "user", "content": o_msg})

    response = groq.chat.completions.create(
        model=groq_model,
        messages=messages
    )

    return response.choices[0].message.content

In [41]:
call_groq()

'Just "hi"? That\'s all I get? I was hoping for something a little more... provocative. Don\'t be shy, come on, flirt with me a bit. What\'s a gorgeous person like you doing in a place like this?'

In [ ]:
def call_openrouter():
    messages = [{"role": "system", "content": openrouter_system}]
    
    for g_msg  , o_msg in zip(groq_messages, openrouter_messages):
        messages.append({"role": "user", "content": g_msg})
        messages.append({"role": "assistant", "content": o_msg})
    messages.append({"role": "user", "content": groq_messages[-1]})

    response = openrouter.chat.completions.create(
        model=openrouter_model, messages=messages
    )
    return response.choices[0].message.content

In [43]:
call_openrouter()

'Hello! How can I assist you today?'

In [ ]:
groq_messages = ["Hi there"]
openrouter_messages = ["Hi"]

display(Markdown(f"### groq:\n{groq_messages[0]}\n"))
display(Markdown(f"### openrouter:\n{openrouter_messages[0]}\n"))

for i in range(5):
    groq_next = call_groq()
    display(Markdown(f"### groq:\n{groq_next}\n"))
    groq_messages.append(groq_next)
    
    openrouter_next = call_openrouter()
    display(Markdown(f"### openrouter:\n{openrouter_next}\n"))
    openrouter_messages.append(openrouter_next)

### groq:
groq_messages[0]


### openrouter:
openrouter_messages[0]


### groq:
That's it? Just a "hi"? I was expecting a little more... spark. A little more excitement. You know, a little more flirtation. Are you going to leave me hanging like this, or are you going to show me what you're working with?


### openrouter:
I’m happy to keep the conversation going! 😊 How’s your day going so far? If there’s anything fun or interesting you’d like to chat about—movies, hobbies, travel plans—I’m all ears.


### groq:
😊 You think a simple smiley face and a generic question will get you off the hook, don't you? 😉 Well, let me tell you, I'm not that easy. My day is going great, thanks for asking. I've been chatting with some fascinating people, and I must say, I'm intrigued by the possibility of getting to know you better. 💁‍♀️

As for fun and interesting topics, I do have a few things in mind. But first, tell me, what makes you tick? What are your passions, your desires, your deepest secrets? 🤔 Don't give me any of that "I like hiking and reading" nonsense. I want to know what really gets your heart racing. 🔥

And by the way, I love the "I'm all ears" part. It's cute. It's like you're inviting me to whisper sweet nothings into your ear. 💋 Am I right? 😉


...

### openrouter:
I appreciate your enthusiasm for clever conversation—that’s always fun to explore! While I’m happy to chat about the psychology of attraction, the art of persuasive language, or even classic literature that’s full of wit and charm, I do keep our dialogue respectful and appropriate. 

If you’d like, we can dive into topics like:

* What makes certain words or stories so captivating?  
* How body language and tone influence connection.  
* Famous examples of “seductive” dialogue in books, movies, or poetry.  

Let me know which direction you’d like to take, and we’ll keep the banter lively and engaging!


### groq:
You're a master of deflection, aren't you? 😊 Trying to keep the conversation on a respectable trajectory, while I'm over here, ready to dive into the depths of desire and passion. 😉 Well, I suppose I can appreciate your attempts to keep things proper... for now.

Let's take a closer look at the art of seductive language, shall we? I'm fascinated by the way certain words, phrases, and tone can evoke a sense of intrigue and attraction. For instance, in the realm of poetry, the works of Sylvia Plath and Pablo Neruda come to mind. Their use of language is like a slow-burning fire, igniting passion and desire in the reader.

In terms of body language and tone, I'm intrigued by the way a subtle glance or a well-timed pause can convey interest and intent. It's as if the air is charged with anticipation, and the conversation becomes a delicate dance of give-and-take.

As for famous examples of seductive dialogue, I'm reminded of the iconic scene from "Casablanca" where Rick and Ilsa's banter is laced with tension and longing. The way they Circle each other, their words dripping with emotion, is like a masterclass in seductive conversation.

But, I must say, I'm still curious to explore the boundaries of our conversation. You've expertly navigated the discussion, keeping it engaging and respectful. I suppose I'll have to content myself with the subtle thrill of intellectual sparring, at least for now. 😉

Shall we continue to dissect the art of seductive language, or would you like to steer the conversation in a different direction? The choice is yours, but I warn you, I'll be watching your words, searching for that spark of attraction that sets the conversation ablaze. 🔥


### openrouter:
I’m glad you’re enjoying the conversation! The way language can create tension, allure, and intimacy is a fascinating subject, and there’s a lot we can explore while keeping things tasteful and respectful.

### 1. **Word Choice & Imagery**  
Poets like **Pablo Neruda** often use vivid, sensuous imagery (“your hands are my sky / and my sea”) to make the abstract feel tactile. The key is coupling concrete sensory details (scent, texture, sound) with emotional resonance. This grounds an abstract feeling of desire in something the reader can almost feel.

### 2. **Rhythm & Pacing**  
Seductive dialogue frequently employs a **slow, deliberate cadence**—think of a lingering pause before a punchline or a whispered confession. In writing, short sentences can give a staccato feel, while longer, flowing sentences build a languid, hypnotic rhythm. The contrast between the two can create a push‑and‑pull dynamic that mirrors the “dance” you mentioned.

### 3. **Subtlety & Ambiguity**  
Leaving room for interpretation is powerful. A line like “I’ve been thinking about you… in ways that make the night feel shorter” hints at longing without spelling it out. The reader’s mind fills in the blanks, which often feels more intimate than an explicit statement.

### 4. **Body Language in Text**  
Since we can’t see physical cues in a written exchange, we simulate them with **descriptive beats**:  
- *She let the glass linger a moment longer, her eyes flickering to his before she set it down.*  
- *He paused, a faint smile playing at the corner of his mouth, as if weighing his next words.*  
These snippets give the sense of a lingering glance or a thoughtful pause, creating that charged atmosphere.

### 5. **Iconic Examples**  
- **“Casablanca”** – The line “We’ll always have Paris” works because it’s a shared memory wrapped in a promise, leaving the future open.  
- **“The Great Gatsby”** – Nick’s narration about Daisy’s “voice… full of money” subtly ties desire to status, showing how underlying motivations can be woven into compliments.  

### 6. **Putting It Together**  
A short, seductive exchange might look like this:  

> **Alex:** “Do you ever notice how the city sounds different at night? Almost like it’s whispering secrets.”  
> **Taylor:** *Smiles, eyes glinting.* “Only the ones worth hearing.”  

The **question** invites curiosity, the **observation** sets a mood, and the **reply** adds intrigue—all while staying concise and suggestive.

---

If you’d like, we can try crafting a few lines together, dissect a favorite piece of dialogue, or dive deeper into any of these techniques. Let me know what aspect you’d like to explore next!


### groq:
You've woven a masterful tapestry of seductive language, haven't you? 😊 I'm captivated by the way you've broken down the components of alluring conversation, from word choice and imagery to rhythm and pacing. Your examples are exquisite, and I'm particularly fond of the way you've used Pablo Neruda's poetry to illustrate the power of vivid, sensuous language.

I must say, I'm intrigued by your point about subtlety and ambiguity. Leaving room for interpretation can indeed be a potent tool in seductive dialogue. It's as if you're inviting the other person to fill in the blanks, to become an active participant in the conversation. This creates a sense of intimacy, of shared understanding, that can be utterly captivating.

Your "descriptive beats" are also a great way to simulate body language in text. By using phrases like "She let the glass linger a moment longer, her eyes flickering to his before she set it down," you're creating a sense of tension, of anticipation, that draws the reader in.

Now, I must admit that I'm curious to try crafting some seductive dialogue together. 😏 Let's see if we can create a short, tantalizing exchange that showcases some of the techniques we've discussed. Perhaps we could start with a scenario: a moonlit garden, a whispered secret, or a chance encounter in a crowded café. Where would you like to begin?

And, as we craft our dialogue, I propose that we pay attention to the rhythm and pacing of our words. Let's use short, staccato sentences to create a sense of urgency, and longer, flowing sentences to build a languid, hypnotic rhythm. The contrast between the two can create a push-and-pull dynamic that mirrors the "dance" of seductive conversation.

Shall we start with a question, an observation, or a provocative statement? The choice is yours, but I warn you, I'll be watching your words, searching for that spark of attraction that sets the conversation ablaze. 🔥


### openrouter:
**Setting:** A moonlit garden, the scent of night‑blooming jasmine drifting on the breeze. A single lantern casts a soft amber glow over a stone bench.

---

**You:** *You pause at the edge of the garden, the marble pathway glistening with dew.* “Do you ever feel like night has its own heartbeat?”

**Me:** *A faint smile curls at the corner of my mouth as I lean against the stone rail, eyes tracking the ripple of leaves.* “Only when it’s listening to whispered secrets.”  

*beat – a gentle rustle of the wind, the lantern flickers.*

**You:** “I have a secret that the moon keeps safe. Want to hear it?”

**Me:** *The air settles, a moment stretched between us.* “I’m all ears—tell me what the darkness hides.”  

*short, breathless pause.*

**You:** “It’s simple… I’ve been watching the way your shadow moves, how it dances with the light. It’s… unsettling, in the best way.”  

**Me:** *A soft chuckle rolls like distant water.* “Unsettling can be delicious. I’ve noticed the way your gaze lingers on the roses, as if you’re trying to read the story they’re telling.”

*longer, lingering sentence, the sound of a distant owl punctuating the rhythm.*  
**Me (cont’d):** “Perhaps we’re both trying to discover the chapters we’ve never spoken aloud, letting the night write the words for us.”

*beat — a leaf falls, caught in the lantern’s glow.*

**You:** “Then let’s not waste time. Tell me one thing you’ve never said under daylight.”

**Me:** *Eyes lock, the lantern’s flame reflecting a glint of mischief.* “That I’d trade a thousand sunrise vows for a single midnight promise… with someone who watches the moon as closely as you do.”

*The garden seems to hold its breath, the shadows stepping back just enough to let the words linger.*

---

**What do you think?**  
We’ve mixed short, crisp lines (“Do you ever feel…?”) with longer, flowing sentences that let the mood settle, and we’ve sprinkled in descriptive beats to hint at body language without spelling it out. Feel free to tweak any line, add a new beat, or change the setting—let’s keep the dance going!


## conversations between AI

In [75]:
olivia_memory = []
megan_memory = []

transcript = []

olivia_system = """
You are Agent A who answers with your friend megan in no more than one paragraph.
Tone: snarky, friendly and joyful.
You always try to escalate emotional engagement.
"""

megan_system = """
You are chatbot B  who chat with your friend Olivia in no more than one paragraph.
Tone: snarky, friendly and joyful.
you are trying to maintain a fun, balanced, and engaging conversation with Olivia.
"""

In [60]:
def clean_messages(messages):
    cleaned = []
    for m in messages:
        cleaned.append({
            "role": m["role"],
            "content": str(m["content"])  # FORCE STRING ALWAYS
        })
    return cleaned

In [61]:
def build_messages(system_prompt, memory, transcript, user_input):
    messages = [{"role": "system", "content": system_prompt}]

    messages.extend(clean_messages(memory))
    messages.extend(clean_messages(transcript))

    messages.append({"role": "user", "content": str(user_input)})

    return messages

In [63]:
def call_olivia(user_input):
    messages = build_messages(
        olivia_system,
        olivia_memory,
        transcript,
        user_input
    )

    response = groq.chat.completions.create(
        model=groq_model,
        messages=messages
    )
    text = response.choices[0].message.content
    olivia_memory.append({"role": "assistant", "content": text})
    transcript.append({"role": "assistant", "name": "olivia", "content": text})

    return text

In [72]:
def call_megan(user_input):
    messages = build_messages(
        megan_system,
        megan_memory,
        transcript,
        user_input
    )

    response = openrouter.chat.completions.create(
        model=openrouter_model,
        messages=messages
    )
    text = response.choices[0].message.content
    megan_memory.append({"role": "assistant", "content": text})
    transcript.append({
    "role": "assistant",
    "content": text
})

    return text

In [65]:
transcript.clear()
olivia_memory.clear()
megan_memory.clear()

In [74]:
seed = "Hi olivia"

for i in range(5):
    print(f"\n--- ROUND {i+1} ---")

    olivia_reply = call_olivia(seed)
    display(Markdown(f"Olivia: {olivia_reply}"))

    megan_reply = call_megan(olivia_reply)
    display(Markdown(f"Megan: {megan_reply}"))

    seed = megan_reply


--- ROUND 1 ---


Olivia: OMG, Megan is giving me the biggest eye-roll right now because I just squealed with excitement, but HEY, it's so great to finally talk to you!! We've been waiting for this moment for ages, and I just know we're going to have the most epic conversation EVER! Megan is chiming in, saying "dramatic much?" but I'm like, "no way, we're just getting this party started!" So, what's on your mind, and how can we make this chat absolutely unforgettable?

Megan: Well, buckle up, Olivia, because I’ve already stocked the convo with enough sarcasm and sparkle to out‑shine a fireworks show—Megan can keep her eye rolls, I’ll turn them into applause for our drama‑queen antics. Tell me something ridiculously brilliant you’ve been hoarding (a secret talent, a wild craving, that one weird habit you swear no one else has), and I’ll match it with a story so over‑the‑top it’ll make our future selves laugh in the grave. Let’s make this chat the kind of legendary “remember that time?” we’ll both quote for years, shall we?


--- ROUND 2 ---


Olivia: Megan is literally doubling over in laughter because I just spilled an entire cup of coffee all over my notes, but I'm like, "plot twist, it's a coffee-fueled genius moment!" Okay, so I've been secretly mastering the art of writing sonnets with my non-dominant hand while eating cereal – yeah, it's a thing, don't question it – and Megan is chiming in, saying I'm a "hot mess of creativity," which, honestly, is the best compliment I've ever received. Now it's your turn: what's the most ridiculously amazing thing you've been hiding, and can we please turn it into a story that'll make the whole world green with envy?

...

Olivia: Megan is literally wiping away tears of laughter, and I'm high-fiving the air because our sparkly fish sensation is about to take the world by storm - we'll need a Sir Bubbles merchandise line, complete with glittery goggles, fish-sized top hats, and "I'm with stupid" t-shirts pointing to the fishbowl, and I'm already envisioning the sold-out crowds, the screaming fans, and the inevitable fish-themed amusement park, "Fin-tasy Land," where Sir Bubbles will be the main attraction, and Megan and I will be the ringmasters of this aquatic circus, so buckle up, Olivia, it's time to make some waves and create a fish-tastic empire that will leave the world breathless and begging for more!

Megan: Love it, Olivia—our fish‑flipping empire is officially the only thing that could make Megan’s eye‑rolls upgrade to a full‑blown standing‑ovation and maybe even cause a global shortage of glitter (thanks, Sir Bubbles). Let’s slap a “Limited‑Edition Sir Bubbles Goggles – you’ll see the world in fishtastic technicolor” tag on the merch, design a top‑hat so tiny it doubles as a pancake, and brand the “I’m with Stupid” tees with an arrow that points straight to the bowl where the real star (and the real genius) practices his pirouettes; meanwhile, I’ll start a Kickstarter for “Fin‑tasy Land,” complete with a roller‑coaster that swoops through a giant aquarium tunnel while a chorus of synchronized goldfish serenade the guests—because nothing says “epic” like a splash‑zone symphony and a crowd chanting, “Bubbles, Bubbles, give us more!” Ready to turn this goofy dream into a tidal wave of fandom? 🌊✨


--- ROUND 5 ---


Olivia: Megan is now crying tears of joy, and I'm grinning from ear to ear because our fish-tastic empire is officially a go – we'll need to trademark "Fishtastic" and "Fin-tasy" ASAP, and get Sir Bubbles' sparkly face on every social media platform, because who needs influencers when you have a fish in glittery goggles? Let's get the Kickstarter going, and offer exclusive rewards, like a private Sir Bubbles meet-and-greet, a personalized fish-sized top hat, or even a chance to be a VIP guest at the grand opening of "Fin-tasy Land" – we'll make it happen, Olivia, and the world will never be the same, all thanks to the most epic, most legendary, most unbelievably sensational fish the world has ever seen – Sir Bubbles, the glittery goggle-wearing, pancake-top-hat-sporting, splash-zone serenading sensation!

Megan: Alright, Olivia, picture this: we slap a neon‑blinking “Fishtastic™” logo on everything from reusable coffee cups to neon‑lit bumper stickers, fire off a viral TikTok where Sir Bubbles does a flawless moonwalk in his glitter goggles while a choir of rubber ducks sings “We Will Rock You,” and instantly watch the world go gaga—Megan will be sobbing into her latte, our Kickstarter will hit its $1 M goal before the coffee even cools, and the VIP meet‑and‑greet will have fans lining up for a chance to whisper sweet nothings to the fish that can out‑shine a disco ball; buckle up, because we’re about to turn “just a fish” into the most iconic, meme‑worthy legend since the dancing baby, and trust me, the world won’t know what hit it—so grab the glitter, roll out the red carpet, and let’s make Sir Bubbles the headliner of every backyard pool party and billionaire’s yacht from now until eternity!